# Same number of runs, twice the information

The budget is fixed: you can afford twelve runs. The doses you put them at is a free choice,
it is usually made by picking round numbers, and it is worth as much as a large increase in
sample size. Twelve points spread evenly over a grid and twelve points chosen for the model
you intend to fit differ by a factor in the determinant of the information matrix — which is
to say, in how tightly the experiment pins the parameters you care about.

Central-composite, Box–Behnken, factorial and fractional-factorial, Latin hypercube — plus
D/A/E-optimal point exchange. On a nonlinear surface optimality is local, so
`bayesian_criterion` averages the criterion over prior draws (review B11); a single draw is the
local special case.

In [ ]:
import numpy as np

from axiom.core import D, Outcome, Treatment
from axiom.surface import (
    Bounds, Criterion, Design, HillKernel, Surface, SurfaceSpec, a_criterion, bayesian_criterion,
    box_behnken, central_composite, d_criterion, defining_relation, e_criterion, equal_spacing,
    fractional_factorial, full_factorial, latin_hypercube, optimal_exchange,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, points

enable();  # every axiom result renders itself from here on

In [ ]:
bounds = Bounds(treatments=("a", "b"), low=(0.0, 0.0), high=(100.0, 40.0))
ccd: Design = central_composite(bounds, alpha="rotatable", center_points=3)
print(ccd.kind, ccd.n, ccd.detail)
print(ccd.as_frame().head())

In [ ]:
b3 = Bounds(treatments=("a", "b", "c"), low=(0.0,) * 3, high=(1.0,) * 3)
print("box-behnken k=3:", box_behnken(b3, center_points=2).n)
print("3^2 factorial:", full_factorial(bounds, 3).n)
ff = fractional_factorial(Bounds(treatments=("A", "B", "C", "D"), low=(0.0,) * 4, high=(1.0,) * 4), generators=("D=ABC",))
print("2^(4-1):", ff.n, ff.detail, defining_relation(("D=ABC",)))
lhs = latin_hypercube(bounds, 12, seed=0)
print("LHS:", lhs.n, "| equal spacing:", equal_spacing(bounds, 4).n)

In [ ]:
def xy(design):
    d = design.doses()
    return (np.asarray(d["a"], dtype=float), np.asarray(d["b"], dtype=float))


fig = points(
    {"central composite": xy(ccd), "latin hypercube": xy(lhs), "equal spacing 4×4": xy(equal_spacing(bounds, 4))},
    title="Three shapes of the same budget",
    subtitle="where each classical design puts its runs in the two-treatment dose space",
    x_title="dose of a (USD)", y_title="dose of b (USD)",
    height=420,
)
caption(fig, "The composite design pushes points to the corners and the axial extremes, "
             "because curvature is estimated at the edges; the hypercube spreads them to "
             "cover a surface whose shape is not assumed. Neither is the default choice — "
             "they answer different questions.")

## Criteria and optimal exchange

Criteria act on the design matrix of the linearized surface. `optimal_exchange` runs a
Fedorov-style point exchange among candidate points; it returns `Unsupported` rather than a
number when no non-singular design is reachable.

In [ ]:
spec = SurfaceSpec(
    name="two_hill",
    treatments=(Treatment(name="a", dimension=D.currency, unit="USD"), Treatment(name="b", dimension=D.currency, unit="USD")),
    outcome=Outcome(name="y", dimension=D.outcome),
    kernels={"a": HillKernel(reference_dose=50.0), "b": HillKernel(reference_dose=20.0)},
)
surface = Surface(spec)
theta = {"alpha": 1.0, "k_a": 50.0, "s_a": 2.0, "beta_a": 10.0, "k_b": 20.0, "s_b": 1.5, "beta_b": 5.0, "sigma": 1.0}
X = surface.linearize(ccd.doses(), theta)
crit: Criterion = "d"
print("D:", round(d_criterion(X), 3), "A:", round(a_criterion(X), 3), "E:", round(e_criterion(X), 4))

In [ ]:
rng = np.random.default_rng(0)
prior_draws = [{**theta, "k_a": float(rng.lognormal(np.log(50), 0.3)), "s_a": float(rng.gamma(4, 0.5))} for _ in range(8)]
candidates = full_factorial(bounds, 7)
naive = equal_spacing(bounds, 3)
opt = optimal_exchange(surface, candidates, n=naive.n, theta_draws=prior_draws, criterion=crit, seed=0)
print("naive   :", round(bayesian_criterion(surface, naive, prior_draws), 3))
print("optimal :", round(bayesian_criterion(surface, opt, prior_draws), 3), "| local (one draw):", round(bayesian_criterion(surface, opt, [theta]), 3))
print(opt.detail)

In [ ]:
fig = points(
    {"equal spacing (naive)": xy(naive), "D-optimal exchange": xy(opt)},
    title="Nine runs, chosen two ways",
    subtitle="the same budget against the same surface — one grid, one exchange over prior draws",
    x_title="dose of a (USD)", y_title="dose of b (USD)",
    height=420,
)
caption(fig, "The exchange puts every run at a corner of the box. The naive grid spends a "
             "third of its budget in the middle, where two saturating curves are flattest and "
             "least distinguishable from each other.")

In [ ]:
scored = {
    "equal spacing 3×3 (9 runs)": naive,
    "latin hypercube (12 runs)": lhs,
    "D-optimal exchange (9 runs)": opt,
    "central composite (11 runs)": ccd,
}
scores = {label: bayesian_criterion(surface, d, prior_draws) for label, d in scored.items()}
table([[label, f"{v:.3f}"] for label, v in scores.items()], headers=("design", "Bayesian D"))
usable = {label: v for label, v in scores.items() if np.isfinite(v)}
fig = compare(
    list(usable), list(usable.values()),
    highlight="D-optimal exchange (9 runs)",
    value_fmt="{:.2f}",
    title="…and what that choice is worth",
    subtitle="Bayesian D-criterion, averaged over eight prior draws of the nonlinear parameters",
    x_title="log |information matrix| (higher is better)",
)
caption(fig, "Nine runs chosen by exchange beat nine on a grid and twelve on a hypercube. "
             "The central composite design is missing from the chart because it scored −∞: on "
             "this nonlinear surface its points make the information matrix singular, which is "
             "the criterion refusing rather than ranking.")

## What this bought you

The classical catalogue when a textbook design is the right answer, and a point exchange
against your own surface and your own prior when it is not — with the criterion averaged over
prior draws rather than evaluated at a guess, because on a nonlinear surface a locally optimal
design is optimal for parameters you do not have yet.

`nbs/design/` prices these designs in nats and in the currency of the decision waiting on them.